In [12]:
import pandas as pd
df = pd.read_csv("bangalore_tech_salaries.csv")
df

,Employee_ID,Role,years_exp,Current_CTC,Previous_CTC,Company,company_TYPE,Skills,Location,Education_Tier,Joining_Year,Work_Mode
0,BLR0065,DS,6,49.4,32.4 LPA,Ola,Unicorn,"JIRA, JavaScript, Spring Boot, Figma, NumPy, A...",HSR Layout,Tier-1,2022,Remote
1,BLR0080,DevOps,0,9.7,NaN,Walmart Labs,MNC,"Tableau, Python, Figma, JavaScript, TensorFlow",Electronic City,2,2024,Remote
2,BLR0166,SDE Backend,6,"3,360,000",24.2 LPA,Postman,Mid-size,"Agile, Excel, GCP, MongoDB, Java, ML",Whitefield,Tier-1,2023,Hybrid
3,BLR0219,SDE Frontend,8,41.4 LPA,30.3,Amazon India,MNC,"PyTorch, Redis, System Design",MG Road,Tier-2,2022,Work from Office
4,BLR0491,Product Lead,5,"3,640,000",30.6 LPA,QuickPay AI,early-stage,"Tableau, Deep Learning, Agile",Whitefield,Tier-2,2023,Onsite
...,...,...,...,...,...,...,...,...,...,...,...,...
1010,BLR0824,Product Designer,2,17.6,12.4,Zoho,Mid-size,"React, SQL, ML, Deep Learning, Java",Jayanagar,Tier-2,2022,WFH
1011,BLR0792,Product Lead,4,₹40.2 LPA,26.6,HCL,MNC,"Excel, Java, Deep Learning, Python, JavaScript",Whitefield,Tier 2,2021,Work from Office
1012,BLR0209,BI Analyst,1,8.6 LPA,6.0 LPA,EdMint,Early-stage,"PyTorch, System Design, MongoDB, JIRA, PowerBI...",NaN,2,2024,Hybrid
1013,BLR0788,PM,2,27.6,17.4,Zomato,Unicorn,"Azure, Kafka, Excel",Bellandur,Tier-3,2023,Remote


In [13]:
#basic inspection
print("Info: ")
df.info()

print("Shape: ")
print(df.shape)

print("Datatypes: ")
print(df.dtypes)

print("Missing values: ")
missing = df.isnull().sum()
print(missing)

print("Duplicate values: ")
duplicates = df.duplicated().sum()
print(duplicates)

#summary
print("-------------------")
print(f"Total rows: {df.shape[0]}")
print(f"Total columns:  {df.shape[1]}")
print(f"Total duplicated: {df.duplicated}")

Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     1015 non-null   object
 1   Role            1015 non-null   object
 2   years_exp       1015 non-null   int64 
 3   Current_CTC     1015 non-null   object
 4   Previous_CTC    815 non-null    object
 5   Company         1015 non-null   object
 6   company_TYPE    1015 non-null   object
 7   Skills          988 non-null    object
 8   Location        995 non-null    object
 9   Education_Tier  1015 non-null   object
 10  Joining_Year    1015 non-null   int64 
 11  Work_Mode       1015 non-null   object
dtypes: int64(2), object(10)
memory usage: 95.3+ KB
Shape: 
(1015, 12)
Datatypes: 
Employee_ID       object
Role              object
years_exp          int64
Current_CTC       object
Previous_CTC      object
Company           object
company_TYPE      object
Skills      

In [25]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()


role_map = {
    "Data": ["ds", "data scientist", "da", "data analyst"],
    "Backend": ["sde backend", "backend developer", "backend engineer"],
    "DevOps": ["devops"],
}

# reverse map
role_reverse = {
    v: k for k, vals in role_map.items() for v in vals
}

df["role_clean"] = (
    df["role"]
    .str.lower()
    .map(role_reverse)
)


company_map = {
    "MNC": ["mnc"],
    "Mid-size": ["mid-size", "mid size"],
    "Early-stage": ["early-stage"],
    "Unicorn": ["unicorn"]
}

company_reverse = {
    v: k for k, vals in company_map.items() for v in vals
}

df["company_type_clean"] = (
    df["company_type"]
    .str.lower()
    .map(company_reverse)
)


edu_map = {
    "Tier 1": ["tier-1", "tier 1", "t1", "1"],
    "Tier 2": ["tier-2", "tier 2", "t2", "2"],
    "Tier 3": ["tier-3", "tier 3", "t3", "3"]
}

edu_reverse = {
    v: k for k, vals in edu_map.items() for v in vals
}

df["education_tier_clean"] = (
    df["education_tier"]
    .str.lower()
    .map(edu_reverse)
)


def convert_ctc(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    # remove currency symbol
    x = x.replace("₹", "")

    # 33,60,000 → 33.6 LPA
    if "," in x and x.replace(",", "").replace(".", "").isdigit():
        return float(x.replace(",", "")) / 100000

    # "32.4 lpa"
    if "lpa" in x:
        return float(x.replace("lpa", "").strip())

    # plain number like "10.0"
    if x.replace(".", "").isdigit():
        return float(x)

    return np.nan

# APPLY (correct indentation)
df["current_ctc_lpa"] = df["current_ctc"].apply(convert_ctc)
df["previous_ctc_lpa"] = df["previous_ctc"].apply(convert_ctc)


df["current_ctc_lpa"] = pd.to_numeric(df["current_ctc_lpa"], errors="coerce")
df["previous_ctc_lpa"] = pd.to_numeric(df["previous_ctc_lpa"], errors="coerce")


df["role_clean"] = df["role_clean"].fillna("Unknown")

# DO NOT fill previous_ctc_lpa


# 9. Drop duplicates
df = df.drop_duplicates()


# 10. OUTPUT
print("\nDTYPES:\n", df.dtypes)

print("\nRole Counts:\n", df["role_clean"].value_counts())

print("\nCompany Type Counts:\n", df["company_type_clean"].value_counts())

print("\nEducation Tier Counts:\n", df["education_tier_clean"].value_counts())


DTYPES:
 employee_id               object
role                      object
years_exp                  int64
current_ctc               object
previous_ctc              object
company                   object
company_type              object
skills                    object
location                  object
education_tier            object
joining_year               int64
work_mode                 object
role_clean                object
company_type_clean        object
education_tier_clean      object
current_ctc_lpa          float64
previous_ctc_lpa         float64
exp_band                category
group_median             float64
group_size                 int64
dtype: object

Role Counts:
 role_clean
Unknown    809
Data       114
Backend     55
DevOps      22
Name: count, dtype: int64

Company Type Counts:
 company_type_clean
MNC            342
Mid-size       257
Unicorn        207
Early-stage    194
Name: count, dtype: int64

Education Tier Counts:
 education_tier_clean
Tier 2    500


/tmp/ipykernel_2927/3244981507.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].str.strip()
/tmp/ipykernel_2927/3244981507.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["role_clean"] = (
/tmp/ipykernel_2927/3244981507.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html

In [19]:
role_ctc_stats = (
    df.groupby("role_clean")["current_ctc_lpa"]
    .agg(["median", "mean", "min", "max"])
    .sort_values(by="median", ascending=False)
)

print(role_ctc_stats)
highest_role = role_ctc_stats.index[0]
lowest_role = role_ctc_stats.index[-1]

print("Highest Paying Role:", highest_role)
print("Lowest Paying Role:", lowest_role)
role_ctc_stats["mean_minus_median"] = (
    role_ctc_stats["mean"] - role_ctc_stats["median"]
)

print(role_ctc_stats)

            median       mean  min   max
role_clean                              
Backend      23.10  24.332727  7.9  55.1
Unknown      22.10  24.564895  5.2  84.4
Data         19.45  22.050877  5.6  73.5
DevOps       17.10  20.140908  9.7  52.6
Highest Paying Role: Backend
Lowest Paying Role: DevOps
            median       mean  min   max  mean_minus_median
role_clean                                                 
Backend      23.10  24.332727  7.9  55.1           1.232727
Unknown      22.10  24.564895  5.2  84.4           2.464895
Data         19.45  22.050877  5.6  73.5           2.600877
DevOps       17.10  20.140908  9.7  52.6           3.040908


In [20]:
backend_df = df[df["role_clean"] == "Backend"].copy()

backend_df["exp_band"] = pd.cut(
    backend_df["years_exp"],
    bins=[0, 1, 3, 5, 100],
    labels=["0-1", "2-3", "4-5", "6+"],
    include_lowest=True
)

exp_ctc = (
    backend_df.groupby("exp_band")["current_ctc_lpa"]
    .median()
    .reset_index()
)

print(exp_ctc)
exp_ctc["growth"] = exp_ctc["current_ctc_lpa"].diff()
print(exp_ctc)

  exp_band  current_ctc_lpa
0      0-1             10.3
1      2-3             20.3
2      4-5             26.5
3       6+             40.4
  exp_band  current_ctc_lpa  growth
0      0-1             10.3     NaN
1      2-3             20.3    10.0
2      4-5             26.5     6.2
3       6+             40.4    13.9


/tmp/ipykernel_2927/4241348554.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  backend_df.groupby("exp_band")["current_ctc_lpa"]


In [21]:
sde_df = df[df["role_clean"].isin(["Backend", "Frontend", "FullStack"])].copy()

skills_to_check = ["aws", "ml", "system design", "kubernetes"]

results = []

for skill in skills_to_check:
    has_skill = sde_df[sde_df["skills"].str.lower().str.contains(skill, na=False)]
    no_skill = sde_df[~sde_df["skills"].str.lower().str.contains(skill, na=False)]

    median_with = has_skill["current_ctc_lpa"].median()
    median_without = no_skill["current_ctc_lpa"].median()

    results.append({
        "skill": skill,
        "with_skill": median_with,
        "without_skill": median_without,
        "premium": median_with - median_without
    })

skill_df = pd.DataFrame(results).sort_values(by="premium", ascending=False)

print(skill_df)

           skill  with_skill  without_skill  premium
1             ml        27.6          21.10     6.50
0            aws        27.1          21.05     6.05
2  system design        23.2          22.55     0.65
3     kubernetes        19.8          23.20    -3.40


In [26]:
#  Q3.5 — Underpaid Professionals (Combined)


group_cols = ["role_clean", "company_type_clean", "exp_band"]


df["exp_band"] = pd.cut(
    df["years_exp"],
    bins=[0, 1, 3, 5, 100],
    labels=["0-1", "2-3", "4-5", "6+"],
    include_lowest=True
)


df["group_median"] = df.groupby(group_cols)["current_ctc_lpa"].transform("median")
df["group_size"] = df.groupby(group_cols)["employee_id"].transform("count")

# Filter reliable groups (>=10 members)
df_filtered = df[df["group_size"] >= 10].copy()

df_filtered["gap"] = df_filtered["current_ctc_lpa"] - df_filtered["group_median"]


underpaid = (
    df_filtered
    .sort_values(by="gap")
    .head(10)
)

# Output
print(underpaid[[
    "employee_id",
    "role_clean",
    "company_type_clean",
    "years_exp",
    "exp_band",
    "current_ctc_lpa",
    "group_median",
    "gap"
]])

    employee_id role_clean company_type_clean  years_exp exp_band  \
606     BLR0879    Unknown            Unicorn          6       6+   
58      BLR0009    Unknown            Unicorn          6       6+   
330     BLR0366    Unknown                MNC          6       6+   
349     BLR0954    Unknown                MNC          6       6+   
718     BLR0980    Unknown                MNC          6       6+   
766     BLR0012    Unknown                MNC          6       6+   
282     BLR0644    Unknown                MNC          6       6+   
339     BLR0912    Unknown            Unicorn          6       6+   
558     BLR0057    Unknown            Unicorn          6       6+   
428     BLR0593    Unknown           Mid-size          6       6+   

     current_ctc_lpa  group_median   gap  
606             29.9          46.4 -16.5  
58              32.2          46.4 -14.2  
330             25.2          38.7 -13.5  
349             25.4          38.7 -13.3  
718             26.8     

/tmp/ipykernel_2927/1100999461.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["group_median"] = df.groupby(group_cols)["current_ctc_lpa"].transform("median")
/tmp/ipykernel_2927/1100999461.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df["group_size"] = df.groupby(group_cols)["employee_id"].transform("count")


In [24]:
line = "=" * 60
sub_line = "-" * 60

print("\n" + line)
print(f"{'BANGALORE TECH SALARY DECODER':^60}")
print(f"{'Built by Shruti Gaikwad | The Unlox Academy':^60}")
print(f"{'2-Hour Live Project':^60}")
print(line)

print(f"{'Dataset : 1,000 Bengaluru tech professionals (synthetic)':<60}")
print(f"{'Period  : 2024 employment snapshot':<60}")

print("\n" + sub_line)
print(f"{'MEDIAN CTC BY ROLE (in LPA)':^60}")
print(sub_line)

role_stats = (
    df.groupby("role_clean")["current_ctc_lpa"]
    .median()
    .sort_values(ascending=False)
)

max_val = role_stats.max()

for role, val in role_stats.items():
    bar = "█" * int((val / max_val) * 20)
    print(f"{role:<20} {bar:<22} {val:>6.1f}")

print("\n" + sub_line)
print(f"{'SDE BACKEND CTC BY EXPERIENCE BAND':^60}")
print(sub_line)

backend_df = df[df["role_clean"] == "Backend"]

exp_ctc = (
    backend_df.groupby("exp_band")["current_ctc_lpa"]
    .median()
    .reset_index()
)

prev = None
for _, row in exp_ctc.iterrows():
    band = str(row["exp_band"])
    val = row["current_ctc_lpa"]

    if prev is None:
        print(f"{band:<15}: {val:>6.1f} LPA median")
    else:
        growth = ((val - prev) / prev) * 100
        print(f"{band:<15}: {val:>6.1f} LPA (+{growth:>3.0f}%)")
    prev = val

print("\n" + sub_line)
print(f"{'SKILL PREMIUM FOR SDEs (median CTC)':^60}")
print(sub_line)
print(f"{'Skill':<15}{'With':>10}{'Without':>12}{'Premium':>12}")

sde_df = df[df["role_clean"].isin(["Backend", "Frontend", "FullStack"])]

skills = ["aws", "ml", "system design", "kubernetes"]

for skill in skills:
    with_s = sde_df[sde_df["skills"].str.lower().str.contains(skill, na=False)]
    without_s = sde_df[~sde_df["skills"].str.lower().str.contains(skill, na=False)]

    m1 = with_s["current_ctc_lpa"].median()
    m2 = without_s["current_ctc_lpa"].median()

    premium = ((m1 - m2) / m2) * 100 if m2 else 0

    print(f"{skill.title():<15}{m1:>10.1f}{m2:>12.1f}{premium:>11.0f}%")

print("\n" + sub_line)
print(f"{'COMPANY-TYPE PREMIUM (SDE Backend)':^60}")
print(sub_line)

company_ctc = (
    backend_df.groupby("company_type_clean")["current_ctc_lpa"]
    .median()
)

unicorn = company_ctc.get("Unicorn", np.nan)

for comp, val in company_ctc.items():
    if comp == "Unicorn":
        print(f"{comp:<15}{val:>8.1f} LPA  <- baseline")
    else:
        pct = ((val - unicorn) / unicorn) * 100 if unicorn else 0
        print(f"{comp:<15}{val:>8.1f} LPA  ({pct:>4.0f}% vs Unicorn)")

print("\n" + sub_line)
print(f"{'TOP 5 MOST-UNDERPAID PROFESSIONALS':^60}")
print(sub_line)

top5 = underpaid.head(5)

for _, row in top5.iterrows():
    print(f"{row['employee_id']:<8} {row['role_clean']:<12} "
          f"{row['company_type_clean']:<10} {int(row['years_exp']):>2} yrs  "
          f"gap: {row['gap']:>6.1f} LPA")

print("\n" + line)
print(f"{'The Unlox Academy Live Project – Salary Decoder':^60}")
print(f"{'© The Unlox Academy':^60}")

print("\nKEY INSIGHTS\n")

# Insights
top_role = role_stats.idxmax()
top_val = role_stats.max()

growth_total = (
    exp_ctc["current_ctc_lpa"].iloc[-1] -
    exp_ctc["current_ctc_lpa"].iloc[0]
)

best_skill = None
best_premium = -999

for skill in skills:
    with_s = sde_df[sde_df["skills"].str.lower().str.contains(skill, na=False)]
    without_s = sde_df[~sde_df["skills"].str.lower().str.contains(skill, na=False)]

    m1 = with_s["current_ctc_lpa"].median()
    m2 = without_s["current_ctc_lpa"].median()

    premium = m1 - m2

    if premium > best_premium:
        best_premium = premium
        best_skill = skill

print(f"1. {top_role} has the highest median CTC at {top_val:.1f} LPA")
print(f"2. Backend salaries grow by {growth_total:.1f} LPA across experience levels")
print(f"3. {best_skill.title()} gives the highest premium (~{best_premium:.1f} LPA)")

print(line)


               BANGALORE TECH SALARY DECODER                
        Built by Shruti Gaikwad | The Unlox Academy         
                    2-Hour Live Project                     
Dataset : 1,000 Bengaluru tech professionals (synthetic)    
Period  : 2024 employment snapshot                          

------------------------------------------------------------
                MEDIAN CTC BY ROLE (in LPA)                 
------------------------------------------------------------
Backend              ████████████████████     23.1
Unknown              ███████████████████      22.1
Data                 ████████████████         19.4
DevOps               ██████████████           17.1

------------------------------------------------------------
             SDE BACKEND CTC BY EXPERIENCE BAND             
------------------------------------------------------------
0-1            :   10.3 LPA median
2-3            :   20.3 LPA (+ 97%)
4-5            :   26.5 LPA (+ 31%)
6+             

/tmp/ipykernel_2927/552754255.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  backend_df.groupby("exp_band")["current_ctc_lpa"]


Key insights
1. Unicorns pay ~30% higher CTC than MNCs for Backend roles, so students should target startups if their goal is higher salary.

2. Backend salaries increase from ~10 LPA (0–1 yrs) to ~35 LPA (6+ yrs), so switching jobs early can boost salary growth faster.

3. System Design gives ~20% higher CTC among SDEs, so focusing on this skill can directly improve job offers.
